In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [3]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is inquiring about various aspects of Lunapolis, including its capital, weather, population of cheese miners, and their potential union actions.\n\n## SUMMARY\nThe capital of the moon is Lunapolis. The weather in Lunapolis is clear, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis. The cheese miners' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='81eb5920-0fe9-473f-bbbb-89e106d44ed7'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='3c45d99f-8dc7-4f26-bd71-800c5678feac'),
              AIMessage(content='If I were Lunapolis’ new pres

In [4]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is inquiring about various aspects of Lunapolis, including its capital, weather, population of cheese miners, and their potential union actions.

## SUMMARY
The capital of the moon is Lunapolis. The weather in Lunapolis is clear, with temperatures ranging from a high of 120°C to a low of -100°C. There are 100,000 cheese miners living in Lunapolis. The cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [5]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [6]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [7]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='a93088d5-05c6-452c-9b12-376f02ca4b29'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='350bc326-7fc2-4247-a723-94f73da9c44f', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='19b8372c-3c18-408b-acb6-939f04652a3e'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='5f810dbd-7a50-4567-b918-bacd7f43817d', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='d359d6ae-62f4-460e-b02d-8fe0aa850df9'),
              AIMessage(content='I can’t read your device’s temperature from here. If overheati

In [8]:
print(response["messages"][-1].content)

I can’t read your device’s temperature from here. If overheating might be the cause, try these steps:

- Let it cool: If it feels hot, power it off, unplug it, and let it sit in a cool, well-ventilated area for 15–30 minutes. Do not try to turn it back on while it’s hot.
- Check for airflow: Make sure vents aren’t blocked by cushions, fabric, or dust. If you can, gently clean dust from the vents with compressed air.
- Inspect the power supply: Try a different outlet and/or a different charger if you have one. Make sure you’re using the correct charger and cable.
- Look for indicators: Do you see any lights or hear beeps when you try to power it on? A certain pattern or color can indicate a fault (e.g., battery, GPU, or motherboard issues).
- Disconnect peripherals: Remove any USB devices, SD cards, external drives, etc., and then try powering on.
- Hard reset / power cycle: Hold the power button for 10–15 seconds to force a shut down, then try turning it on again after a minute.
- If i